# C73：固定三臂身份／相位可行性运行包（卡74）

已采纳新增64维同步形状约束，目标0.5、rho<1/16；净32和A不变。
用户GPU运行：P0/S_base/S_ID全部fresh D，3D/18个九帧reset E，0 FFT/DiT/梯度/新host。
六个独立clip每臂，G只复用同臂D（含失败）。完整target与连续context分别评价。
36物理向量、1656身份槽（3正/1653负），全部失败/未定义保留；全拒读不是成功。
先持久保存全部y/s/rho/status才后评角色/真几何；码成员不等于相位认证，不恢复完整gap。
质量6表1086帧/1080邻接及3preclip。载荷层未采纳，不能评价全32退步或独立FPR。

直接Run all默认从现有dev分支的普通URL自动下载配套源码ZIP，随后执行固定流程，无需手动上传ZIP、改参数或新增代码。SOURCE_LOCATION也支持可用源码目录、本地ZIP或普通URL。
任何attempt/科学残留存在时，后续Run all只读保存结果，不补跑模型/媒体/失败槽。
无hash/manifest/版本/Git干净状态或资源硬上限。agent仅CPU核验，未运行本GPU notebook。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-C73-Identity-V1/fixed-seen-host-v1')
def execution_artifacts(output):
    output = Path(output)
    records = ("attempt.json", "result.json", "checkpoint.json", "ledger.json", "protocol.json",
               "construction.json", "numeric_completion.json", "evaluation.json", "native_audit.json")
    found = [name+suffix for name in records for suffix in ("", ".tmp")
             if (output/(name+suffix)).exists() or (output/(name+suffix)).is_symlink()]
    for folder in ("blind", "features", "media", "quality", "books"):
        root = output/folder
        if root.is_dir():
            found.extend(str(p.relative_to(output)) for p in root.rglob("*")
                         if p.is_file() or p.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if EXISTING_ARTIFACTS:
    print('Existing attempt: saved records only', EXISTING_ARTIFACTS)
else:
    import stat, urllib.request, zipfile
    SOURCE_LOCATION = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/dev/local-joint-state-payload-v1/notebooks/c73_identity_v1_portable_source.zip'
    source = Path(SOURCE_LOCATION)
    if source.is_dir():
        SOURCE_ROOT = source
    else:
        archive = source
        if SOURCE_LOCATION.startswith(('https://','http://')):
            archive = Path('/content/c73_downloaded_source.zip')
            urllib.request.urlretrieve(SOURCE_LOCATION, archive)
        SOURCE_ROOT = Path('/content/c73_identity_source')
        SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                target = (SOURCE_ROOT/info.filename).resolve()
                if not target.is_relative_to(SOURCE_ROOT.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError('unsafe source archive path')
            bundle.extractall(SOURCE_ROOT)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/c73_identity_v1_run.py').is_file():
        raise FileNotFoundError('actual C73 runner missing from available source')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    print('Available editable source:', SOURCE_ROOT)


依赖及一次固定运行：仅修复实际导入/媒体依赖问题。已有attempt时跳过源码下载、依赖安装及物理调用。

In [ ]:

import subprocess, sys

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/"setup.log"
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode

    # Copied from the already-used A/P2 setup; no A import/build dependency.
    def ensure_imports(probe, packages, label):
        import re
        module_packages = {package.replace('-', '_'): package for package in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], ('Recheck ' if repaired else 'Probe ')+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1) in module_packages:
                package = module_packages[missing.group(1)]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' failed without a known targeted repair; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, 'Repair '+label+': '+action)

    ensure_imports('import numpy,torch', ['numpy', 'torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
                   ['diffusers', 'accelerate', 'safetensors', 'huggingface-hub'], 'Wan VAE imports')
    def ensure_media_tools():
        import shutil
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            command(['apt-get', '-qq', 'update'], 'Prepare media dependency')
            command(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], 'Install FFmpeg')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe still unavailable; see setup.log')
    ensure_media_tools()

    with (OUTPUT/"runner.log").open("w") as log:
        process = subprocess.Popen([sys.executable, "-u", "-m",
            "experiments.wan_state_clock.c73_identity_v1_run"],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    import time
    try:
        while process.poll() is None:
            time.sleep(30)
            ledger_path = OUTPUT/"ledger.json"
            if ledger_path.exists():
                ledger = json.loads(ledger_path.read_text())
                returned = sum(row["status"] == "RETURNED" for row in ledger.values())
                active = [name for name, row in ledger.items() if row["status"] == "STARTED"]
                print(f"Completed core units: {returned}/{len(ledger)}; active: {active}")
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Stopped; saved STARTED calls remain unresolved and will not be replayed.")
        raise
    print("Runner return code:", process.returncode)
else:
    print("Existing attempt: displaying saved records; no physical calls repeated.")
print("Output:", OUTPUT)


保存记录：失败/中断/未进入槽均保留；再次Run all仅显示，不恢复执行。

In [ ]:

saved = {}
for name in ('result.json','checkpoint.json','ledger.json','numeric_completion.json','evaluation.json','attempt.json'):
    path = OUTPUT/name
    if path.is_file():
        try:
            saved[name] = json.loads(path.read_text())
        except (OSError, ValueError) as exc:
            print('Unreadable saved record:', name, type(exc).__name__, str(exc))
r = saved.get('result.json',saved.get('checkpoint.json',{}))
print('Execution:', r.get('status'), 'Fixed denominators:', r.get('fixed_denominator'))
print('Calls:', r.get('counts'))
evaluation = saved.get('evaluation.json',{})
print('Positive retained / planned3:', evaluation.get('retained_positive'))
print('All-ID false attributions:', evaluation.get('false_attributions'))
print('Positive losses/failures:', evaluation.get('positive_not_retained'))
print('Actual construction:', r.get('construction'))
print('FP32/native old22 audit:', r.get('native_audit'))
print('Quality:', r.get('quality'), 'Preclip:', r.get('preclip'))
for name, row in saved.get('ledger.json',{}).items():
    if row.get('status') != 'RETURNED':
        print('Retained failed/unentered/started slot:', name, row)
print('Code membership is not phase authentication; no gap path, FPR or full32 payload claim. FFT0.')
print('Output:', OUTPUT)
